# W1-M2 실습 2 — G1 해부: lesson §5의 표를 직접 재생성하기

lesson.md `§5`(G1 모델 읽기) 전체를 **자료를 믿지 말고 직접 뽑아서** 확인하는 스크립트입니다.

이 실습의 교육 포인트는 숫자 자체가 아니라 **"모델 스펙은 문서가 아니라 모델에서 읽는다"** 는 습관입니다.
menagerie는 계속 갱신되고, 회사가 쓰는 MJCF는 또 다를 수 있습니다(lesson §5.3의 팀 확인 항목).
그때마다 이 스크립트를 돌리면 됩니다.

확인할 것:

1. 세 변형(`scene.xml` / `scene_mjx.xml` / `scene_with_hands.xml`)의 `nq/nv/nu/nbody/njnt/ngeom/nkey` ·
   키프레임 이름 · `timestep` · `iterations` 비교 — lesson §5.2
2. `nq = 7 + n_{hinge}`, `nv = 6 + n_{hinge}` 를 **assert로 검증** — lesson §3.4
3. 관절 30개 전체를 `id / 이름 / type / qpos adr / dof adr / range`로 출력 — lesson §5.4
4. **세 주소 체계가 어긋나는 것을 계산으로 검증**:
   `ctrl id = i` ↔ `qpos adr = i+7` ↔ `qvel adr = i+6` ↔ `jnt id = i+1`
5. 액추에이터 `kp`/`kv` 추출 후 **유효관성 역산** $M_{eff} = (k_v/2\zeta)^2/k_p$ — lesson §5.5, 퀴즈 7번
6. 센서 4개와 `sensordata[12]` — lesson §5.7
7. 킨매틱 트리를 `body_parentid`로 순회해 들여쓰기 출력
   → 같은 폴더의 `g1_kinematic_tree_worksheet.excalidraw` 빈칸을 채울 재료

**GPU 불필요. 렌더도 하지 않습니다. CPU에서 수 초.**
`--csv`를 주면 관절표를 `joints_g1.csv`로 저장합니다.

In [ ]:
from __future__ import annotations

import os

# 이 스크립트는 렌더를 하지 않지만, mujoco import 전에 백엔드를 고정해두는 습관을 유지한다 (lesson §6.2).
os.environ.setdefault("MUJOCO_GL", "egl")

import argparse  # noqa: E402
import csv  # noqa: E402
import sys  # noqa: E402
import unicodedata  # noqa: E402
from pathlib import Path  # noqa: E402

import mujoco  # noqa: E402
import numpy as np  # noqa: E402

MODULE_ID = "W1-M2"

# lesson §5.5 — <position kp="500" dampratio="1" inheritrange="1"/>
DAMPRATIO = 1.0  # 임계감쇠 zeta

## 0. 경로 유틸

**모델 경로 규약** (세 스크립트 공통):

1. `--menagerie <경로>` 인자가 있으면 그것
2. 없으면 환경변수 `MENAGERIE_PATH`
3. 그것도 없으면 리포 루트 기준 `repos/mujoco_menagerie`

```bash
git clone --depth 1 https://github.com/google-deepmind/mujoco_menagerie.git repos/mujoco_menagerie
```
(약 2.3 GB. `repos/`는 gitignore 대상입니다.)

In [ ]:
_ROOT_MARKERS = ("course", "docs", "CLAUDE.md")


def find_repo_root() -> Path:
    """리포 루트 디렉토리를 찾는다 (스크립트/노트북 양쪽에서 동작)."""
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd().resolve()
    for cand in (start, *start.parents):
        if all((cand / m).exists() for m in _ROOT_MARKERS):
            return cand
    return start


def artifacts_dir() -> Path:
    out = find_repo_root() / "artifacts" / MODULE_ID
    out.mkdir(parents=True, exist_ok=True)
    return out


def resolve_menagerie(arg: str | None = None) -> Path:
    """mujoco_menagerie 경로를 결정한다 (--menagerie > MENAGERIE_PATH > repos/mujoco_menagerie)."""
    cand = arg or os.environ.get("MENAGERIE_PATH") or str(find_repo_root() / "repos" / "mujoco_menagerie")
    p = Path(cand).expanduser().resolve()
    if not (p / "unitree_g1").is_dir():
        raise SystemExit(
            f"[에러] mujoco_menagerie를 찾지 못했습니다: {p}\n"
            "  해결: git clone --depth 1 https://github.com/google-deepmind/mujoco_menagerie.git "
            "repos/mujoco_menagerie\n"
            "  또는: --menagerie <경로> / 환경변수 MENAGERIE_PATH 지정"
        )
    return p


def g1_scene(menagerie: Path, variant: str = "scene.xml") -> Path:
    """unitree_g1/<variant> 경로. 로드할 것은 g1.xml이 아니라 scene*.xml (lesson §5.1)."""
    p = menagerie / "unitree_g1" / variant
    if not p.is_file():
        raise SystemExit(f"[에러] 파일이 없습니다: {p}")
    return p

In [ ]:
def _dwidth(s: str) -> int:
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)


def _pad(s: str, width: int, align: str = "left") -> str:
    gap = max(0, width - _dwidth(s))
    if align == "right":
        return " " * gap + s
    if align == "center":
        left = gap // 2
        return " " * left + s + " " * (gap - left)
    return s + " " * gap


def print_table(headers: list[str], rows: list[list[str]], aligns: list[str] | None = None) -> None:
    """한글 폭을 고려한 간단한 표 출력."""
    aligns = aligns or ["left"] * len(headers)
    widths = [
        max(_dwidth(headers[i]), *(_dwidth(r[i]) for r in rows)) if rows else _dwidth(headers[i])
        for i in range(len(headers))
    ]
    print(" | ".join(_pad(h, w, "center") for h, w in zip(headers, widths)))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(" | ".join(_pad(c, w, a) for c, w, a in zip(r, widths, aligns)))


def names_of(m: mujoco.MjModel, objtype, n: int) -> list[str]:
    """id -> 이름 리스트. 이름이 없으면 '(unnamed)'."""
    return [mujoco.mj_id2name(m, objtype, i) or "(unnamed)" for i in range(n)]

## 1. 세 변형 비교 — lesson §5.2

`unitree_g1/`에는 씬 파일이 세 개 있습니다.

| 파일 | 무엇 |
|---|---|
| `scene.xml` | classic 29 DoF. **디버깅·sim2sim 검증용** |
| `scene_mjx.xml` | MJX용. 콜리전 단순화 + 게인 하향 + `timestep` 2배 |
| `scene_with_hands.xml` | 손 포함 43 DoF |

아래 표가 lesson §5.2의 표와 **한 칸이라도 다르면** menagerie가 갱신된 것입니다.
그때는 lesson이 아니라 이 출력이 맞습니다 — 이 실습의 요점이 그것입니다.

In [ ]:
VARIANTS = ["scene.xml", "scene_mjx.xml", "scene_with_hands.xml"]

# lesson §5.2 실측표 (2026-08-01 기준). 다르면 경고를 찍되 실행은 계속한다.
LESSON_EXPECT = {
    "scene.xml": dict(nq=36, nv=35, nu=29, nbody=31, njnt=30, ngeom=72, nkey=1,
                      timestep=0.002, iterations=100),
    "scene_mjx.xml": dict(nq=36, nv=35, nu=29, nbody=31, njnt=30, ngeom=63, nkey=2,
                          timestep=0.004, iterations=5),
    "scene_with_hands.xml": dict(nq=50, nv=49, nu=43, nbody=45, njnt=44, ngeom=101, nkey=1,
                                 timestep=0.002, iterations=100),
}


def variant_stats(m: mujoco.MjModel) -> dict:
    return dict(
        nq=m.nq, nv=m.nv, nu=m.nu, nbody=m.nbody, njnt=m.njnt, ngeom=m.ngeom, nkey=m.nkey,
        timestep=float(m.opt.timestep), iterations=int(m.opt.iterations),
        ls_iterations=int(m.opt.ls_iterations),
        solver=mujoco.mjtSolver(m.opt.solver).name,
        integrator=mujoco.mjtIntegrator(m.opt.integrator).name,
        keys=names_of(m, mujoco.mjtObj.mjOBJ_KEY, m.nkey),
    )


def compare_variants(menagerie: Path) -> dict[str, dict]:
    """세 변형을 모두 로드해 비교표를 출력하고 lesson §5.2와 대조한다."""
    print("\n=== [1] 세 변형 비교 (lesson §5.2) ===")
    stats: dict[str, dict] = {}
    for v in VARIANTS:
        m = mujoco.MjModel.from_xml_path(str(g1_scene(menagerie, v)))
        stats[v] = variant_stats(m)

    fields = ["nq", "nv", "nu", "nbody", "njnt", "ngeom", "nkey",
              "timestep", "iterations", "ls_iterations", "solver", "integrator"]
    rows = [[f] + [f"{stats[v][f]:g}" if isinstance(stats[v][f], float) else str(stats[v][f])
                   for v in VARIANTS] for f in fields]
    rows.append(["키프레임"] + [", ".join(stats[v]["keys"]) for v in VARIANTS])
    print_table(["항목"] + VARIANTS, rows, aligns=["left", "right", "right", "right"])

    # lesson과 대조
    mismatches = []
    for v in VARIANTS:
        for k, expected in LESSON_EXPECT[v].items():
            got = stats[v][k]
            if not np.isclose(got, expected):
                mismatches.append(f"{v}.{k}: 실측 {got} != lesson {expected}")
    if mismatches:
        print("\n  ⚠️ lesson §5.2와 다릅니다 (menagerie가 갱신됐을 수 있습니다):")
        for msg in mismatches:
            print(f"     - {msg}")
        print("     -> 이 출력이 맞습니다. lesson 표를 갱신하고 docs/progress.md에 기록하세요.")
    else:
        print("\n  ✅ lesson §5.2 표와 전부 일치합니다.")

    print("\n  읽는 법 (lesson §3.5):")
    print("   - MJX 버전은 timestep 2배 · iterations 1/20 · ngeom 축소. 정확도를 처리량과 맞바꾼 설정.")
    print("   - 키프레임 이름이 다르다: classic 'stand' vs MJX 'home','knees_bent'.")
    print("     mj_resetDataKeyframe(m, d, 0)을 두 모델에 그대로 쓰면 초기 자세가 서로 다르다.")
    print("     -> 인덱스가 아니라 mj_name2id(..., mjOBJ_KEY, '이름')으로 찾을 것.")
    return stats

## 2. `nq` ≠ `nv` 를 손으로 유도 — lesson §3.4

```
nq = 7 (free joint: xyz 3 + quat wxyz 4) + n_hinge
nv = 6 (free joint: 선속도 3 + 각속도 3)  + n_hinge
nu =                                        n_hinge   (자유 관절은 구동되지 않음)
```

세 식이 동시에 성립하는지 `assert`로 확인합니다. 깨지면 모델 구성이 위 가정과 다른 것입니다
(예: free joint가 2개, 또는 slide 관절이 섞임).

In [ ]:
def check_dof_arithmetic(m: mujoco.MjModel, label: str) -> None:
    """자유 관절 1개 + hinge n개 가정이 성립하는지 검증한다 (lesson §3.4)."""
    types = m.jnt_type
    n_free = int((types == mujoco.mjtJoint.mjJNT_FREE).sum())
    n_hinge = int((types == mujoco.mjtJoint.mjJNT_HINGE).sum())
    n_slide = int((types == mujoco.mjtJoint.mjJNT_SLIDE).sum())
    n_ball = int((types == mujoco.mjtJoint.mjJNT_BALL).sum())

    print(f"\n=== [2] nq != nv 유도 — {label} (lesson §3.4) ===")
    print(f"  관절 구성: free {n_free} · hinge {n_hinge} · slide {n_slide} · ball {n_ball}"
          f"  (합계 njnt={m.njnt})")
    print(f"  nq = 7*{n_free} + 1*{n_hinge} = {7 * n_free + n_hinge}   (실측 {m.nq})")
    print(f"  nv = 6*{n_free} + 1*{n_hinge} = {6 * n_free + n_hinge}   (실측 {m.nv})")
    print(f"  nu = {n_hinge}  (자유 관절에는 액추에이터가 없다)        (실측 {m.nu})")
    assert m.nq == 7 * n_free + n_hinge, "nq 유도 실패"
    assert m.nv == 6 * n_free + n_hinge, "nv 유도 실패"
    assert m.nu == n_hinge, "nu != hinge 수 — 액추에이터가 붙지 않은 관절이 있다"
    print(f"  nq - nv = {m.nq - m.nv} = free joint 개수. 쿼터니언(4)이 각속도(3)보다 한 칸 길기 때문.")
    print("  -> 그래서 qpos += qvel*dt 를 직접 쓰면 안 된다. mj_integratePos / mj_differentiatePos 사용.")

## 3. 관절 30개 전체 + 세 주소 체계 — lesson §5.4

이 절이 첫 주에 가장 많은 시간을 잡아먹는 함정입니다.

```
관절 이름 ── jnt id  = i + 1     (자유 관절이 id 0)
             qpos adr = i + 7    (자유 관절이 앞에서 7칸)
             qvel adr = i + 6    (자유 관절이 앞에서 6칸)
             ctrl id  = i        (자유 관절은 액추에이터가 없음)
```

이 규칙이 실제로 성립하는지 **모델에서 읽은 값으로 검증**합니다.

> **규칙은 깨질 수 있습니다.** `scene.xml`(29 DoF)에서는 29개 전부 성립하지만,
> `scene_with_hands.xml`(43 DoF)에서는 손가락 관절 4개가 규칙을 벗어납니다 —
> MJCF의 `<actuator>` 선언 순서와 body 트리의 관절 순서가 다르기 때문입니다.
> 직접 확인해보세요: `python 02_g1_inspect.py --variant scene_with_hands.xml`
>
> 그래서 이 검증은 실패해도 예외를 던지지 않고 **어느 관절이 어긋났는지 표로 보여줍니다.**
> 규칙을 외우는 것이 아니라 "이 규칙은 모델에 따라 깨진다"는 것을 아는 것이 목적입니다.

In [ ]:
CHAIN_RULES = (
    ("left_leg", ("left_hip", "left_knee", "left_ankle")),
    ("right_leg", ("right_hip", "right_knee", "right_ankle")),
    ("waist", ("waist",)),
    ("left_arm", ("left_shoulder", "left_elbow", "left_wrist")),
    ("right_arm", ("right_shoulder", "right_elbow", "right_wrist")),
)


def chain_of(name: str) -> str:
    """관절 이름을 6개 체인 중 하나로 분류 (excalidraw 워크시트의 가지와 대응)."""
    for chain, prefixes in CHAIN_RULES:
        if any(name.startswith(p) for p in prefixes):
            return chain
    return "other"


def joint_rows(m: mujoco.MjModel) -> list[dict]:
    """관절 전체를 dict 리스트로. CSV 저장과 표 출력이 같은 소스를 쓴다."""
    jnames = names_of(m, mujoco.mjtObj.mjOBJ_JOINT, m.njnt)
    anames = names_of(m, mujoco.mjtObj.mjOBJ_ACTUATOR, m.nu)
    act_by_joint = {}
    for aid, an in enumerate(anames):
        # <position joint="X"/> 이므로 actuator_trnid[aid,0]이 관절 id
        act_by_joint[int(m.actuator_trnid[aid, 0])] = aid

    bnames = names_of(m, mujoco.mjtObj.mjOBJ_BODY, m.nbody)

    rows = []
    for jid in range(m.njnt):
        jtype = mujoco.mjtJoint(m.jnt_type[jid]).name.replace("mjJNT_", "").lower()
        aid = act_by_joint.get(jid)
        lo, hi = (float(m.jnt_range[jid, 0]), float(m.jnt_range[jid, 1]))
        limited = bool(m.jnt_limited[jid])
        kp = kv = m_eff = None
        if aid is not None:
            kp = float(m.actuator_gainprm[aid, 0])  # deep-dive §11: gainprm[0] = kp
            kv = float(-m.actuator_biasprm[aid, 2])  # biasprm = [0, -kp, -kv]
            if kp > 0:
                # eq.(3)  k_v = 2*zeta*sqrt(k_p * M_eff)  ->  M_eff = (k_v / (2*zeta))^2 / k_p
                m_eff = (kv / (2.0 * DAMPRATIO)) ** 2 / kp
        body = int(m.jnt_bodyid[jid])
        rows.append(dict(
            jnt_id=jid,
            name=jnames[jid],
            type=jtype,
            qpos_adr=int(m.jnt_qposadr[jid]),
            dof_adr=int(m.jnt_dofadr[jid]),
            ctrl_id=aid,
            range_lo=lo if limited else None,
            range_hi=hi if limited else None,
            kp=kp,
            kv=kv,
            m_eff=m_eff,
            chain=chain_of(jnames[jid]),
            body=bnames[body],
            parent_body=bnames[int(m.body_parentid[body])],
        ))
    return rows


def print_joint_table(rows: list[dict]) -> None:
    print("\n=== [3] 관절 전체 목록 (lesson §5.4) ===")
    trows = []
    for r in rows:
        rng = "—" if r["range_lo"] is None else f"[{r['range_lo']:.3f}, {r['range_hi']:.3f}]"
        trows.append([
            str(r["jnt_id"]),
            r["name"],
            r["type"],
            str(r["qpos_adr"]),
            str(r["dof_adr"]),
            "—" if r["ctrl_id"] is None else str(r["ctrl_id"]),
            rng,
            r["chain"],
        ])
    print_table(
        ["jnt id", "이름", "type", "qpos adr", "dof adr", "ctrl id", "range (rad)", "체인"],
        trows,
        aligns=["right", "left", "left", "right", "right", "right", "right", "left"],
    )
    print("  좌우 대칭 관절의 range는 부호가 뒤집혀 있다 (예: left_hip_roll vs right_hip_roll).")
    print("  좌우 궤적을 복사할 때 부호를 뒤집어야 한다.")


def check_index_alignment(m: mujoco.MjModel, rows: list[dict]) -> list[dict]:
    """세 주소 체계 정렬표를 계산으로 검증한다 (lesson §5.4).

    ⚠️ 규칙이 깨져도 예외를 던지지 않습니다. **깨지는 모델이 실제로 존재**하기 때문입니다
    (`scene_with_hands.xml`의 손가락 관절 — 아래 출력 참조). 규칙이 깨졌다는 사실 자체가
    "손으로 세지 말고 이름으로 조회하라"는 §5.4 조언의 물증입니다.
    """
    print("\n=== [4] 세 주소 체계 정렬 검증 (lesson §5.4) ===")
    actuated = [r for r in rows if r["ctrl_id"] is not None]
    bad = []
    for r in actuated:
        i = r["ctrl_id"]
        exp = (i + 1, i + 7, i + 6)
        got = (r["jnt_id"], r["qpos_adr"], r["dof_adr"])
        if exp != got:
            r = dict(r, expected=exp, got=got)
            bad.append(r)

    # 대표 관절 몇 개만 표로
    picks = ["left_knee_joint", "waist_yaw_joint", "left_elbow_joint", "right_wrist_yaw_joint"]
    by_name = {r["name"]: r for r in rows}
    trows = []
    for p in picks:
        if p not in by_name:
            continue
        r = by_name[p]
        trows.append([r["name"], str(r["ctrl_id"]), str(r["jnt_id"]), str(r["qpos_adr"]), str(r["dof_adr"])])
    print_table(
        ["관절", "ctrl id = i", "jnt id = i+1", "qpos adr = i+7", "qvel adr = i+6"],
        trows,
        aligns=["left", "right", "right", "right", "right"],
    )

    if not bad:
        print(f"  ✅ 구동 관절 {len(actuated)}개 전부 규칙을 만족합니다.")
    else:
        print(f"  ❗ 구동 관절 {len(actuated)}개 중 **{len(bad)}개가 규칙을 벗어납니다.**")
        brows = [[
            r["name"],
            str(r["ctrl_id"]),
            f"{r['expected'][0]} / {r['expected'][1]} / {r['expected'][2]}",
            f"{r['got'][0]} / {r['got'][1]} / {r['got'][2]}",
        ] for r in bad]
        print_table(
            ["관절", "ctrl id", "규칙이 예측하는 jnt/qpos/qvel", "실제 jnt/qpos/qvel"],
            brows, aligns=["left", "right", "right", "right"],
        )
        print("  원인: MJCF의 <actuator> 선언 순서와 body 트리의 관절 순서가 다릅니다.")
        print("        `ctrl id + 7 = qpos adr` 같은 산술 규칙은 **모델이 그렇게 짜여 있을 때만** 성립합니다.")
        print("  ❗❗ 이것이 바로 lesson §5.4가 '손으로 세지 말라'고 하는 이유의 실물입니다.")
        print("       이 규칙을 믿고 코드를 짜면 손가락 두 개가 조용히 뒤바뀐 채 돌아갑니다.")

    print("  실무 조회법 — 어느 모델에서도 안전합니다:")
    print("    jid  = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_JOINT, 'left_knee_joint')")
    print("    qadr = m.jnt_qposadr[jid];  vadr = m.jnt_dofadr[jid]")
    print("    aid  = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_ACTUATOR, 'left_knee_joint')")
    print("    jid_of_act = m.actuator_trnid[aid, 0]   # 액추에이터가 실제로 무는 관절 id")
    return bad

## 4. 액추에이터와 유효관성 역산 — lesson §5.5, 퀴즈 7번

MJCF의 `g1` 기본 클래스는 `<position kp="500" dampratio="1" inheritrange="1"/>` 한 줄입니다.
컴파일되면 `gainprm[0]=kp`, `biasprm=[0, -kp, -kv]`가 되고 액추에이터가 내는 힘은

$$
f \;=\; k_p\,(\texttt{ctrl} - q) \;-\; k_v\,\dot q \tag{1}
$$

**즉 `data.ctrl[i]`는 토크가 아니라 목표 관절각[rad]입니다.**

`dampratio="1"`(임계감쇠)이므로 MuJoCo가 관절마다 `kv`를 따로 계산합니다.

$$
k_v = 2\,\zeta\sqrt{k_p M_{\text{eff}}}
\quad\Longleftrightarrow\quad
M_{\text{eff}} = \frac{(k_v/2\zeta)^2}{k_p} \tag{2}
$$

이 역산값에는 `armature`(반사 관성)가 포함돼 있고, 실제 $M(q)$는 자세에 따라 변하지만
`kv`는 **컴파일 시점에 한 번 정해지는 상수**입니다. 그 사실 자체가 sim2real 소재입니다.

In [ ]:
def print_actuator_table(m: mujoco.MjModel, rows: list[dict], top: int | None = None) -> None:
    """kp/kv/M_eff 표. lesson §5.5의 네 관절이 여기 포함된다."""
    print("\n=== [5] 액추에이터 게인과 유효관성 역산 (lesson §5.5, 퀴즈 7번) ===")
    gaintype = mujoco.mjtGain(m.actuator_gaintype[0]).name if m.nu else "-"
    biastype = mujoco.mjtBias(m.actuator_biastype[0]).name if m.nu else "-"
    print(f"  gaintype={gaintype}  biastype={biastype}   (position 서보 = fixed gain + affine bias)")
    fr = m.actuator_forcerange
    unlimited = int((fr[:, 0] == 0).sum() & (fr[:, 1] == 0).sum()) if m.nu else 0
    print(f"  forcerange 예시: {fr[0]}  -> [0,0]이면 **토크 제한 없음** (lesson §5.6 갭 씨앗 #1)")
    print(f"  ctrlrange == jnt_range 인가(inheritrange=1): "
          f"{np.allclose(m.actuator_ctrlrange, m.jnt_range[1:1 + m.nu])}")

    act = [r for r in rows if r["ctrl_id"] is not None]
    picks = ["left_hip_pitch_joint", "left_knee_joint", "left_shoulder_pitch_joint", "left_elbow_joint"]
    show = [r for r in act if r["name"] in picks] if top is None else act[:top]
    trows = [[r["name"], f"{r['kp']:g}", f"{r['kv']:.8f}", f"{r['m_eff']:.4f}"] for r in show]
    print_table(
        ["액추에이터", "kp", "kv (실측)", "M_eff = (kv/2ζ)²/kp  [kg·m²]"],
        trows,
        aligns=["left", "right", "right", "right"],
    )
    kvs = np.array([r["kv"] for r in act])
    meffs = np.array([r["m_eff"] for r in act])
    print(f"  전체 {len(act)}개: kv ∈ [{kvs.min():.3f}, {kvs.max():.3f}],  "
          f"M_eff ∈ [{meffs.min():.4f}, {meffs.max():.4f}] kg·m²  (최대/최소 비 {meffs.max() / meffs.min():.0f}배)")
    print("  -> 몸 전체를 흔드는 고관절과 아래팔만 흔드는 팔꿈치의 차이. 전 관절에 같은 kv를 주면")
    print("     어떤 관절은 과감쇠, 어떤 관절은 부족감쇠가 된다. dampratio가 그걸 자동으로 맞춘다.")
    print(f"  참고: 서보의 자연진동수 w_n = sqrt(kp/M_eff) [rad/s] 도 관절마다 다르다 —")
    lo_w = np.sqrt(500.0 / meffs.max()) / (2 * np.pi)
    hi_w = np.sqrt(500.0 / meffs.min()) / (2 * np.pi)
    print(f"     kp=500 기준 대략 {lo_w:.1f} ~ {hi_w:.1f} Hz. 03_g1_sin_wave.py의 추종 오차가 이 대역과 관련된다.")


def compare_actuator_gains(menagerie: Path) -> None:
    """classic vs MJX 액추에이터 게인 비교 (lesson §3.5 표)."""
    print("\n=== [6] classic vs MJX 액추에이터 게인 (lesson §3.5) ===")
    trows = []
    for v in ("scene.xml", "scene_mjx.xml"):
        m = mujoco.MjModel.from_xml_path(str(g1_scene(menagerie, v)))
        kp = m.actuator_gainprm[:, 0]
        kv = -m.actuator_biasprm[:, 2]
        trows.append([
            v,
            f"{kp.min():g} ~ {kp.max():g}",
            f"{kv.min():.3f} ~ {kv.max():.3f}",
            f"{m.opt.timestep:g}",
            str(m.opt.iterations),
        ])
    print_table(["모델", "kp 범위", "kv 범위", "timestep", "iterations"], trows,
                aligns=["left", "right", "right", "right", "right"])
    print("  MJX는 kp를 낮추고 kv를 전 관절 동일 상수로 고정했다.")
    print("  뻣뻣한 PD는 큰 timestep에서 발산하기 쉽다 — 이산 PD의 안정 영역이 게인×dt에 걸린다.")
    print("  같은 로봇의 MJCF가 두 벌이고 물리 파라미터가 다르다 = sim2sim의 축소판 (lesson §3.5).")

## 5. 센서 — lesson §5.7

**관절 엔코더 센서가 정의돼 있지 않습니다.** 관절각은 `data.qpos[7:]`, 각속도는 `data.qvel[6:]`을
직접 읽어야 합니다. 시뮬은 골반의 절대 위치·자세까지 참값으로 주지만 실기는 그것을 모릅니다 —
관측 벡터를 조립할 때 **어느 값이 실기에서도 얻어지는가**를 매번 따져야 하는 이유입니다.

In [ ]:
def print_sensors(m: mujoco.MjModel, d: mujoco.MjData) -> None:
    print("\n=== [7] 센서 (lesson §5.7) ===")
    snames = names_of(m, mujoco.mjtObj.mjOBJ_SENSOR, m.nsensor)
    rows = []
    for sid, sname in enumerate(snames):
        rows.append([
            str(sid),
            sname,
            mujoco.mjtSensor(m.sensor_type[sid]).name.replace("mjSENS_", ""),
            str(int(m.sensor_dim[sid])),
            str(int(m.sensor_adr[sid])),
        ])
    print_table(["id", "이름", "type", "dim", "sensordata adr"], rows,
                aligns=["right", "left", "left", "right", "right"])
    total = int(m.sensor_dim.sum())
    print(f"  nsensor={m.nsensor},  sensordata 길이={d.sensordata.shape[0]} (= 합계 {total})")
    assert d.sensordata.shape[0] == total
    print("  관절 엔코더 센서는 없다 -> 관절각은 data.qpos[7:], 각속도는 data.qvel[6:]에서 직접 읽는다.")

## 6. 킨매틱 트리 — excalidraw 워크시트의 재료

`body_parentid`를 순회해 body 트리를 들여쓰기로 그립니다.
각 body에 달린 관절의 `ctrl id`/`qpos adr`를 옆에 붙여두면,
같은 폴더의 `g1_kinematic_tree_worksheet.excalidraw` 빈칸을 그대로 채울 수 있습니다.

In [ ]:
def print_kinematic_tree(m: mujoco.MjModel, max_depth: int | None = None) -> None:
    print("\n=== [8] 킨매틱 트리 (body_parentid 순회) ===")
    bnames = names_of(m, mujoco.mjtObj.mjOBJ_BODY, m.nbody)
    jnames = names_of(m, mujoco.mjtObj.mjOBJ_JOINT, m.njnt)
    anames = names_of(m, mujoco.mjtObj.mjOBJ_ACTUATOR, m.nu)
    act_by_joint = {int(m.actuator_trnid[a, 0]): a for a in range(m.nu)}

    children: dict[int, list[int]] = {i: [] for i in range(m.nbody)}
    for b in range(1, m.nbody):
        children[int(m.body_parentid[b])].append(b)

    def walk(b: int, depth: int) -> None:
        if max_depth is not None and depth > max_depth:
            return
        js = []
        for k in range(int(m.body_jntnum[b])):
            jid = int(m.body_jntadr[b]) + k
            aid = act_by_joint.get(jid)
            tag = f"{jnames[jid]}(jnt{jid}, qpos{int(m.jnt_qposadr[jid])}"
            tag += f", ctrl{aid})" if aid is not None else ", ctrl—)"
            js.append(tag)
        joint_str = "  ←  " + " , ".join(js) if js else ""
        mass = float(m.body_mass[b])
        print(f"  {'    ' * depth}{'└─ ' if depth else ''}{bnames[b]}  [{mass:.3f} kg]{joint_str}")
        for c in children[b]:
            walk(c, depth + 1)

    for c in children[0]:
        walk(c, 0)
    print(f"  총 {m.nbody - 1}개 body (world 제외), 총질량 {m.body_mass.sum():.3f} kg")
    print("  -> 이 트리를 g1_kinematic_tree_worksheet.excalidraw의 빈칸에 옮겨 적으세요.")

## 7. CSV 저장

W1-M1의 `players.csv`와 같은 취지입니다 — **모델이 바뀌면 다시 뽑는 파일**입니다.

In [ ]:
CSV_FIELDS = ["jnt_id", "name", "type", "qpos_adr", "dof_adr", "ctrl_id",
              "range_lo", "range_hi", "kp", "kv", "m_eff", "chain", "body", "parent_body"]


def save_csv(rows: list[dict], path: Path) -> Path:
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        w.writeheader()
        for r in rows:
            w.writerow({k: ("" if r[k] is None else r[k]) for k in CSV_FIELDS})
    return path

## 8. 실행

In [ ]:
def _in_notebook() -> bool:
    try:
        from IPython import get_ipython  # type: ignore

        if get_ipython() is not None:
            return True
    except Exception:
        pass
    argv0 = Path(sys.argv[0]).name.lower() if sys.argv else ""
    return argv0 == "" or "ipykernel" in argv0 or "jupyter" in argv0 or "colab" in argv0


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    p = argparse.ArgumentParser(description="W1-M2 실습 2: G1 모델 해부 (lesson §5)")
    p.add_argument("--menagerie", default=None,
                   help="mujoco_menagerie 경로 (기본: $MENAGERIE_PATH 또는 <repo>/repos/mujoco_menagerie)")
    p.add_argument("--variant", default="scene.xml", choices=VARIANTS,
                   help="상세 해부 대상 (기본 scene.xml)")
    p.add_argument("--csv", action="store_true", help="관절표를 artifacts/W1-M2/joints_g1.csv 로 저장")
    p.add_argument("--all-actuators", action="store_true", help="[5] 표에 액추에이터 전체를 출력")
    p.add_argument("--smoke", action="store_true", help="세 변형 비교 + 인덱스 검증만 (표 출력 축약)")
    if argv is None:
        argv = [] if _in_notebook() else sys.argv[1:]
    return p.parse_args(argv)


def main(argv: list[str] | None = None) -> None:
    args = parse_args(argv)
    menagerie = resolve_menagerie(args.menagerie)

    print("=" * 92)
    print(f" W1-M2 실습 2 — G1 해부 {'(smoke)' if args.smoke else ''}")
    print(f" menagerie: {menagerie}")
    print(f" mujoco {mujoco.__version__}")
    print("=" * 92)

    compare_variants(menagerie)

    path = g1_scene(menagerie, args.variant)
    m = mujoco.MjModel.from_xml_path(str(path))
    d = mujoco.MjData(m)
    # 키프레임은 인덱스가 아니라 이름으로 찾는다 (lesson §3.5)
    kname = "stand" if "stand" in names_of(m, mujoco.mjtObj.mjOBJ_KEY, m.nkey) else None
    kid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_KEY, kname) if kname else 0
    if m.nkey:
        mujoco.mj_resetDataKeyframe(m, d, kid)
    mujoco.mj_forward(m, d)
    print(f"\n  상세 해부 대상: {path.name}  (키프레임 '{kname or names_of(m, mujoco.mjtObj.mjOBJ_KEY, m.nkey)[kid]}' 적용)")
    print(f"  골반 높이 qpos[2] = {d.qpos[2]:.3f} m,  쿼터니언 qpos[3:7] = {np.array2string(d.qpos[3:7], precision=3)}")

    check_dof_arithmetic(m, path.name)

    rows = joint_rows(m)
    if not args.smoke:
        print_joint_table(rows)
    check_index_alignment(m, rows)
    print_actuator_table(m, rows, top=m.nu if args.all_actuators else None)
    compare_actuator_gains(menagerie)
    print_sensors(m, d)
    if not args.smoke:
        print_kinematic_tree(m)

    if args.csv:
        out = save_csv(rows, artifacts_dir() / "joints_g1.csv")
        print(f"\n[저장] {out}   ({len(rows)}행 · 컬럼 {', '.join(CSV_FIELDS)})")
        print("  이 CSV를 보며 g1_kinematic_tree_worksheet.excalidraw의 빈칸을 채우세요.")

    print("\n" + "-" * 92)
    print(" 다음: 03_g1_sin_wave.py — ctrl에 sin파를 넣어 실제로 움직이게 한다")
    print("-" * 92)

In [ ]:
if __name__ == "__main__":
    main()